In [1]:
# inlegalbert_bilstm_mha_crf_synonym_aug.py
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# BASE: inlegalbert_bilstm_mha_crf_rrc_v2.py  (all anti-overfitting settings kept)
#
# NEW IN THIS FILE — PROTECTED LEGAL SYNONYM AUGMENTATION:
# ─────────────────────────────────────────────────────────
#  A. LEGAL_TERMS protection corpus
#     A hand-crafted set of domain-critical tokens (court names, party roles,
#     procedural terms, Latin maxims) that must NEVER be replaced, so that
#     rhetorical meaning is preserved.
#
#  B. Synonym replacement (WordNet)
#     For each sentence, up to α × len(words) non-protected, non-stop-word
#     tokens are replaced with a randomly chosen WordNet synonym.
#     Case-style is preserved (Title-case → Title-case).
#
#  C. Targeted rare-class augmentation  (augment_docs)
#     Sentences belonging to RARE classes (freq ≤ RARE_THRESHOLD) are
#     duplicated RARE_AUG_FACTOR−1 extra times (with synonym perturbation).
#     Common-class sentences are left untouched (COMMON_AUG_FACTOR = 1).
#     The number of augmented copies is capped so that rare-class counts
#     never exceed the dataset median (avoids over-representation).
#     Augmentation is applied to the TRAIN split ONLY.
#
#  D. Post-augmentation reporting
#     A before/after label-frequency table is printed and saved to CSV.
#     All other outputs (plots, CSVs, checkpoints) are identical to v2.
#
# ALL anti-overfitting settings from v2 are RETAINED unchanged:
#   early stopping (patience=10), BERT freeze (bottom 8 layers),
#   layer-wise LR decay (0.9/layer), dropout=0.4, weight_decay=0.05,
#   reduced LSTM sizes (sent=128, ctx=64), aux CE loss, label smoothing.
#

import os, json, random, time, re
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

import nltk
# Download required NLTK resources (silent if already present)
for _pkg in ("wordnet", "stopwords", "omw-1.4"):
    try:
        nltk.download(_pkg, quiet=True)
    except Exception:
        pass
from nltk.corpus import wordnet, stopwords as nltk_stopwords

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_synaugv2_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 60          # early stopping fires well before this
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# ── BERT freeze / layer-wise LR decay ─────────────────────
BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

# ── Sentence-level BiLSTM ──────────────────────────────────
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1

# ── Context BiLSTM ────────────────────────────────────────
CTX_LSTM_HIDDEN = 64
CTX_LSTM_LAYERS = 2

# ── Auxiliary loss ─────────────────────────────────────────
AUX_CE_WEIGHT   = 0.2
LABEL_SMOOTHING = 0.1

# ── Early stopping ─────────────────────────────────────────
ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ══════════════════════════════════════════════════════════
# SYNONYM AUGMENTATION CONFIG   [NEW]
# ══════════════════════════════════════════════════════════
# α: fraction of tokens eligible for replacement per sentence
ALPHA             = 0.10

# How many augmented copies to generate per rare-class sentence
# (total copies = 1 original + (RARE_AUG_FACTOR - 1) augmented)
RARE_AUG_FACTOR   = 4

# Common-class sentences are NOT duplicated (factor = 1 means no extra copy)
COMMON_AUG_FACTOR = 1

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":          "InLegalBERT Encoder",
        "sent_bilstm":   "Sentence BiLSTM",
        "mha_pooling":   "Multi-Head Attn Pooling",
        "ctx_bilstm":    "Context BiLSTM",
        "classifier":    "Classifier Head",
        "crf":           "CRF",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({"Component": name, "Trainable Params": trainable,
                     "Frozen Params": frozen, "Total Params": trainable + frozen})

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({"Component": "── TOTAL ──",
                 "Trainable Params": total_trainable,
                 "Frozen Params": total_frozen,
                 "Total Params": total_trainable + total_frozen})

    print("\n" + "=" * 72)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT+BiLSTM+MHA+CRF  Syn-Aug)")
    print("=" * 72)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 72)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 72)
        print(f"  {r['Component']:<30} {r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} {r['Total Params']:>12,}")
    print("=" * 72)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD, tag=""):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    header = f"📊 Label frequencies {tag} (threshold ≤ {threshold*100:.0f}%):"
    print(f"\n{header}")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════════════════════
# ██████╗  PROTECTED LEGAL SYNONYM AUGMENTATION  [NEW]  ████████████████████
# ═══════════════════════════════════════════════════════════════════════════

# ── A. Protected legal-term corpus ────────────────────────────────────────
# These tokens carry domain-critical meaning for rhetorical roles and must
# NEVER be replaced by a synonym. Replacing "petitioner" with "suppliant"
# or "ratio" with "proportion" would destroy label semantics.
LEGAL_TERMS: set = {
    # Party / role designations
    "petitioner", "respondent", "appellant", "appellee",
    "plaintiff", "defendant", "accused", "complainant",
    "prosecution", "defence", "defense",
    # Court / institution names
    "court", "tribunal", "bench", "judge", "justice",
    "magistrate", "commissioner", "registrar",
    "supreme court", "high court", "district court",
    "sessions court", "civil court", "criminal court",
    # Procedural concepts
    "preamble", "facts", "issue", "argument", "analysis",
    "ratio", "ratio decidendi", "obiter", "obiter dicta",
    "ruling", "verdict", "judgment", "judgement", "order",
    "decree", "injunction", "stay", "writ",
    "appeal", "revision", "review", "petition",
    "bail", "remand", "custody", "parole",
    "conviction", "acquittal", "sentence", "punishment",
    "charge", "indictment", "complaint", "fir",
    # Latin maxims
    "habeas corpus", "mandamus", "certiorari", "prohibition",
    "quo warranto", "inter alia", "prima facie", "res judicata",
    "sub judice", "mens rea", "actus reus",
    # Statute / document references
    "section", "article", "clause", "schedule", "provision",
    "act", "code", "regulation", "rule", "ordinance",
    "constitution", "amendment",
    # Evidence / procedural
    "evidence", "witness", "testimony", "exhibit",
    "examination", "cross-examination", "affidavit",
    "deposition", "summons", "notice", "warrant",
    # Precedent
    "precedent", "relied", "followed", "distinguished",
    "overruled", "affirmed", "reversed", "remanded",
}

# Load English stop-words once at module level
_STOP_WORDS: set = set(nltk_stopwords.words("english"))


# ── B. Synonym lookup (legal-protected) ───────────────────────────────────
def get_synonyms(word: str) -> list:
    """
    Return WordNet synonyms for `word`, excluding:
      • the word itself
      • any term in LEGAL_TERMS
      • any English stop-word
      • multi-word phrases (contain spaces / underscores after normalisation)
    """
    w = word.lower()
    if w in LEGAL_TERMS or w in _STOP_WORDS or len(w) <= 2:
        return []

    synonyms = set()
    for syn in wordnet.synsets(w):
        for lemma in syn.lemmas():
            candidate = lemma.name().replace("_", " ").lower()
            # skip multi-word, protected, stop-words, and the word itself
            if (
                " " not in candidate
                and candidate != w
                and candidate not in LEGAL_TERMS
                and candidate not in _STOP_WORDS
                and len(candidate) > 2
            ):
                synonyms.add(candidate)

    return list(synonyms)


def synonym_replacement(sentence: str, alpha: float = ALPHA) -> str:
    """
    Replace up to α × len(words) eligible tokens with a random WordNet synonym.
    Preserves original capitalisation style (ALL-CAPS, Title-case, lower-case).
    """
    words    = sentence.split()
    n_words  = len(words)
    if n_words == 0:
        return sentence

    n_replace = max(1, int(n_words * alpha))

    # Indices of tokens eligible for replacement
    eligible = [
        i for i, w in enumerate(words)
        if w.lower() not in LEGAL_TERMS
        and w.lower() not in _STOP_WORDS
        and len(w) > 2
        and re.match(r"^[a-zA-Z]+$", w)   # skip numbers, punctuation
    ]
    if not eligible:
        return sentence

    n_replace = min(n_replace, len(eligible))
    to_replace = random.sample(eligible, n_replace)

    new_words = words[:]
    for pos in to_replace:
        original = words[pos]
        syns     = get_synonyms(original)
        if not syns:
            continue
        replacement = random.choice(syns)

        # Preserve capitalisation style
        if original.isupper():
            replacement = replacement.upper()
        elif original.istitle():
            replacement = replacement.capitalize()

        new_words[pos] = replacement

    return " ".join(new_words)


# ── C. Targeted rare-class augmentation ───────────────────────────────────
def augment_docs(docs: list, rare_ids: set) -> list:
    """
    Augment the training corpus by synonym-perturbing rare-class sentences.

    Strategy
    ────────
    For each sentence in a document:
      • If its label is RARE:
          generate min(needed_to_reach_median, RARE_AUG_FACTOR−1) extra
          copies, each with independent synonym replacement.
      • If its label is COMMON:
          no duplication (COMMON_AUG_FACTOR = 1).

    The "needed" cap is computed against the CURRENT running count so that
    the loop is self-correcting: as rare classes grow, fewer copies are made
    for later documents, preventing over-representation.

    Returns a new list of (sents, labels) tuples; the original list is
    not modified.
    """
    print("\n📝 Applying targeted synonym augmentation (TRAIN only) …")

    # Running label counts (updated as we generate copies)
    label_counts  = Counter(lbl for _, labs in docs for lbl in labs)
    median_count  = int(np.median(list(label_counts.values())))

    before_counts = dict(label_counts)   # snapshot for reporting

    new_docs = []
    total_added = 0

    for sents, labs in docs:
        new_sents, new_labs = [], []

        for sent, lab in zip(sents, labs):
            # Always keep the original sentence
            new_sents.append(sent)
            new_labs.append(lab)

            if lab in rare_ids:
                # How many MORE copies do we still need to approach median?
                needed     = max(0, median_count - label_counts[lab])
                aug_factor = min(needed, RARE_AUG_FACTOR - 1)
            else:
                aug_factor = COMMON_AUG_FACTOR - 1   # = 0

            for _ in range(aug_factor):
                aug_sent = synonym_replacement(sent, alpha=ALPHA)
                new_sents.append(aug_sent)
                new_labs.append(lab)
                label_counts[lab] += 1
                total_added       += 1

        new_docs.append((new_sents, new_labs))

    after_counts = Counter(lbl for _, labs in new_docs for lbl in labs)

    # ── Print before / after table ────────────────────────────────────────
    print(f"\n   {'Label':<20} {'Before':>8} {'After':>8} {'Added':>7}")
    print("   " + "-" * 48)
    for lbl in LABELS:
        lid = label2id[lbl]
        b   = before_counts.get(lid, 0)
        a   = after_counts.get(lid, 0)
        delta = a - b
        flag  = " ▲" if delta > 0 else ""
        print(f"   {lbl:<20} {b:>8,} {a:>8,} {delta:>7,}{flag}")
    print(f"\n   Total sentences added : {total_added:,}")
    print(f"   Total sentences now   : {sum(after_counts.values()):,}\n")

    return new_docs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                       torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L, B  = batch[0]["input_ids"].shape[1], len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query      = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn    = self.attn_drop(F.softmax(attn, dim=-1))
        context = torch.matmul(attn, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL  (unchanged from v2)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2     # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim=sent_out_dim, num_heads=mha_heads, dropout=mha_dropout
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2       # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        n_total = len(self.bert.encoder.layer)
        print(f"\n❄️  BERT frozen : embeddings + layers 0–{n_freeze-1}")
        print(f"🔥 BERT trainable: layers {n_freeze}–{n_total-1} + pooler\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                          lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid      = flat_mask.sum(-1) > 0

        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)

        token_embs       = self.dropout(token_embs)
        lstm_out, _      = self.sent_bilstm(token_embs)
        lstm_out         = self.dropout(lstm_out)
        pad_mask         = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs        = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs        = self.sent_layer_norm(sent_vecs)
        sent_vecs        = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss   = self.ce_loss(
                emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2)
            )
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    def _f1(av): return f1_score(all_trues, all_preds, average=av, zero_division=0)
    def _pr(av): return precision_score(all_trues, all_preds, average=av, zero_division=0)
    def _rc(av): return recall_score(all_trues, all_preds, average=av, zero_division=0)

    present_rare = [r for r in rare_ids if r in all_trues]
    rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                          average="macro", zero_division=0) if present_rare else 0.0
    rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0) if present_rare else 0.0
    rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                              average="macro", zero_division=0) if present_rare else 0.0

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                               average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                      average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                   average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    return {
        "macro_f1":           _f1("macro"),   "micro_f1":           _f1("micro"),
        "weighted_f1":        _f1("weighted"),
        "macro_precision":    _pr("macro"),   "micro_precision":    _pr("micro"),
        "weighted_precision": _pr("weighted"),
        "macro_recall":       _rc("macro"),   "micro_recall":       _rc("micro"),
        "weighted_recall":    _rc("weighted"),
        "rare_f1":            rare_f1,        "rare_precision":     rare_prec,
        "rare_recall":        rare_rec,
        "per_class_metrics":  per_class_metrics,
        "accuracy":           accuracy_score(all_trues, all_preds),
        "cls_report":         classification_report(str_trues, str_preds,
                                                     labels=LABELS, digits=4,
                                                     zero_division=0),
        "cm":                 confusion_matrix(str_trues, str_preds, labels=LABELS),
        "all_preds":          all_preds,
        "all_trues":          all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER  (unchanged from v2, apart from constructor print)
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        # BERT pooler
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        # Encoder layers with top-down LR decay
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth  = (n_layers - 1) - i
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params,
                    "lr": BERT_LR * (BERT_LR_DECAY ** depth),
                    "weight_decay": WEIGHT_DECAY,
                })
        # Head
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, mask, types, labels, lengths in loader:
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, mask, types, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):
        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )

        early_stopper = EarlyStopping()
        history, best_f1, best_state = [], -1.0, None
        total_start, actual_epochs   = time.time(), 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            run_loss, n_steps, nan_steps = 0.0, 0, 0
            t0 = time.time()
            optimizer.zero_grad()

            for step, (ids, mask, types, labels, lengths) in enumerate(train_loader):
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, mask, types, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                run_loss += loss.item(); n_steps += 1

            # flush tail
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time   = time.time() - t0
            avg_loss     = run_loss / max(1, n_steps)
            val_loss     = self.compute_val_loss(dev_dataset)
            val_m        = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan={nan_steps}]" if nan_steps else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                f"val_acc: {val_m['accuracy']:.4f} | "
                f"time: {epoch_time:.1f}s | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{nan_info}"
            )

            history.append({
                "epoch":                  epoch,
                "train_loss":             avg_loss,
                "val_loss":               val_loss,
                "val_accuracy":           val_m["accuracy"],
                "val_macro_f1":           val_m["macro_f1"],
                "val_micro_f1":           val_m["micro_f1"],
                "val_weighted_f1":        val_m["weighted_f1"],
                "val_rare_f1":            val_m["rare_f1"],
                "val_macro_precision":    val_m["macro_precision"],
                "val_micro_precision":    val_m["micro_precision"],
                "val_weighted_precision": val_m["weighted_precision"],
                "val_rare_precision":     val_m["rare_precision"],
                "val_macro_recall":       val_m["macro_recall"],
                "val_micro_recall":       val_m["micro_recall"],
                "val_weighted_recall":    val_m["weighted_recall"],
                "val_rare_recall":        val_m["rare_recall"],
                "epoch_train_time_s":     epoch_time,
                "nan_steps":              nan_steps,
                "timestamp":              datetime.utcnow().isoformat(),
            })

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_m["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

            if early_stopper.step(val_m["macro_f1"]):
                print(f"\n⏹  Early stopping after epoch {epoch}.\n")
                break

        total_time = time.time() - total_start
        print(f"\n⏱  Total : {total_time/60:.2f} min  ({actual_epochs} epochs)")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump({
                "total_training_time_s":   total_time,
                "total_training_time_min": total_time / 60,
                "avg_epoch_time_s":        total_time / max(1, actual_epochs),
                "num_epochs_run":          actual_epochs,
            }, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        t0 = time.time()

        with torch.no_grad():
            for ids, mask, types, labels, lengths in loader:
                ids, mask, types = (ids.to(self.device), mask.to(self.device),
                                    types.to(self.device))
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, mask, types, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].cpu().numpy().tolist())

        if measure_inference_time:
            elapsed    = time.time() - t0
            n_sents    = len(all_trues)
            infer_info = {
                "split": split_name, "n_documents": n_samples,
                "n_sentences": n_sents,
                "total_inference_time_s": elapsed,
                "latency_per_document_ms": elapsed / max(1, n_samples) * 1000,
                "latency_per_sentence_ms": elapsed / max(1, n_sents) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, elapsed),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {elapsed:.2f}s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f}ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
            "augmentation": {
                "method":             "protected_synonym_replacement",
                "alpha":              ALPHA,
                "rare_aug_factor":    RARE_AUG_FACTOR,
                "common_aug_factor":  COMMON_AUG_FACTOR,
                "legal_terms_count":  len(LEGAL_TERMS),
            },
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        ax.set_title("Training vs Validation Loss  (Synonym Aug)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1  (Synonym Aug)")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].plot(epochs, hist_df["train_loss"], label="Train", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"],
                   color="steelblue", alpha=0.8)
            ax.axhline(hist_df["epoch_train_time_s"].mean(), color="red",
                       linestyle="--",
                       label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s")
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y"); plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix  (Synonym Aug)"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1  (Synonym Aug)"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 70)
    print("FINAL RESULTS  (InLegalBERT+BiLSTM+MHA+CRF  +  Synonym Aug)")
    print("=" * 70)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 70)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 70)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 70)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 70)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4 layers) → Sentence BiLSTM(128) "
          "→ MHA(4-head) → Context BiLSTM(64) → Linear → CRF + Aux-CE")
    print("\nAnti-overfitting settings (v2, unchanged):")
    print(f"  Dropout={DROPOUT}  WD={WEIGHT_DECAY}  "
          f"Freeze={BERT_FREEZE_LAYERS} layers  LR-decay={BERT_LR_DECAY}")
    print(f"  Aux CE weight={AUX_CE_WEIGHT}  label_smoothing={LABEL_SMOOTHING}")
    print(f"  Early stopping patience={ES_PATIENCE}")
    print("\nSynonym augmentation settings:")
    print(f"  α (replacement rate)  : {ALPHA}")
    print(f"  Rare aug factor       : {RARE_AUG_FACTOR}x copies per rare sentence")
    print(f"  Common aug factor     : {COMMON_AUG_FACTOR}x  (no duplication)")
    print(f"  Protected terms count : {len(LEGAL_TERMS)}\n")

    # ── Load data ─────────────────────────────────────────────────────────
    print("Loading JSONL files …")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    # ── Rare-class detection (on original train) ──────────────────────────
    rare_labels, rare_ids, label_freqs_before = detect_rare_classes(
        train_docs, tag="[BEFORE augmentation]"
    )

    # Save before-augmentation frequencies
    freq_df_before = pd.DataFrame([
        {"label": l, "frequency_before": label_freqs_before[l],
         "is_rare": l in rare_labels}
        for l in LABELS
    ])

    # ── Synonym augmentation (TRAIN ONLY) ─────────────────────────────────
    print("=" * 65)
    print("  PROTECTED LEGAL SYNONYM AUGMENTATION  (TRAIN split only)")
    print("=" * 65)
    train_docs_aug = augment_docs(train_docs, set(rare_ids))

    # ── Rare-class detection (after augmentation — for reporting) ─────────
    _, _, label_freqs_after = detect_rare_classes(
        train_docs_aug, tag="[AFTER augmentation]"
    )

    # Merge before/after frequency CSV
    freq_df = freq_df_before.copy()
    freq_df["frequency_after"] = [label_freqs_after[l] for l in LABELS]
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)
    print(f"Saved label frequency CSV to {OUT_DIR}/label_frequencies.csv")
    print("=" * 65 + "\n")

    # ── Tokenizer + Datasets ──────────────────────────────────────────────
    print("Loading tokenizer …")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    # Augmented train, original dev/test
    train_dataset = RRCDataset(train_docs_aug, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,       tokenizer)
    test_dataset  = RRCDataset(test_docs,      tokenizer)

    print(f"\n  Train docs (aug) : {len(train_dataset)}")
    print(f"  Dev docs         : {len(dev_dataset)}")
    print(f"  Test docs        : {len(test_dataset)}")

    # ── Model ─────────────────────────────────────────────────────────────
    print("\nInitialising model …")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training (max {NUM_EPOCHS} epochs, "
          f"early-stop patience={ES_PATIENCE}) …")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids  = rare_ids,
        tokenizer = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    # ── Load best checkpoint ──────────────────────────────────────────────
    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    # ── Dev evaluation ────────────────────────────────────────────────────
    print("\nEvaluating on Dev …")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy={dev_metrics['accuracy']:.4f}  "
          f"Macro-F1={dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1={dev_metrics['rare_f1']:.4f}")
    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF  +  Synonym Augmentation\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])
    trainer.save_confusion_matrix(dev_metrics["cm"],           "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    # ── Test evaluation ───────────────────────────────────────────────────
    print("\nEvaluating on Test …")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy={test_metrics['accuracy']:.4f}  "
          f"Macro-F1={test_metrics['macro_f1']:.4f}  "
          f"Rare-F1={test_metrics['rare_f1']:.4f}")
    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF  +  Synonym Augmentation\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])
    trainer.save_confusion_matrix(test_metrics["cm"],            "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        pd.DataFrame([
            {"label": lbl, "is_rare": lbl in rare_labels,
             "f1":        mets["per_class_metrics"][lbl]["f1"],
             "precision": mets["per_class_metrics"][lbl]["precision"],
             "recall":    mets["per_class_metrics"][lbl]["recall"]}
            for lbl in LABELS
        ]).to_csv(os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT+BiLSTM+MHA+CRF  +  Synonym Aug",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
            "total_params":     total_trainable + total_frozen,
        },
        "augmentation": {
            "method":            "protected_synonym_replacement",
            "alpha":             ALPHA,
            "rare_aug_factor":   RARE_AUG_FACTOR,
            "common_aug_factor": COMMON_AUG_FACTOR,
            "legal_terms_count": len(LEGAL_TERMS),
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(dev_metrics, test_metrics,
                        total_train_time=total_train_time,
                        total_trainable=total_trainable,
                        total_frozen=total_frozen)

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-4 layers) → Sentence BiLSTM(128) → MHA(4-head) → Context BiLSTM(64) → Linear → CRF + Aux-CE

Anti-overfitting settings (v2, unchanged):
  Dropout=0.4  WD=0.05  Freeze=8 layers  LR-decay=0.9
  Aux CE weight=0.2  label_smoothing=0.1
  Early stopping patience=10

Synonym augmentation settings:
  α (replacement rate)  : 0.1
  Rare aug factor       : 4x copies per rare sentence
  Common aug factor     : 1x  (no duplication)
  Protected terms count : 101

Loading JSONL files …
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequencies [BEFORE augmentation] (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA           

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen : embeddings + layers 0–7
🔥 BERT trainable: layers 8–11 + pooler


MODEL PARAMETER SUMMARY  (InLegalBERT+BiLSTM+MHA+CRF  Syn-Aug)
  Component                           Trainable     Frozen        Total
------------------------------------------------------------------------
  InLegalBERT Encoder                28,942,080 80,540,160  109,482,240
  Sentence BiLSTM                     1,314,816          0    1,314,816
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        264,192          0      264,192
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
────────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        30,728,016 80,540,160  111,268,176

Starting training (max 60 epochs, early-stop patience=10) …
Epoch 001/60 | train_loss: 318.0944 | val_loss: 214.5753 | val_macro_f